# Fine-tuning `riskAwareEvaluationFunction` (Q6)
Mục tiêu: Sử dụng Optuna để tìm kiếm các trọng số tối ưu nhất cho hàm `riskAwareEvaluationFunction` mới, ít tham số hơn (đã bỏ `w_capsule_dist` do không còn sử dụng).
- Bản đồ: `smallClassic`
- Số lượng ghost: 1
- Đánh giá: Trung vị (Median) của 30 ván chơi.

In [4]:
import os
import re
import subprocess
import sys
import optuna
import statistics
import plotly

# Cấu hình đường dẫn
ROOT_DIR = os.path.dirname(os.path.abspath('')) # file notebook nằm trong thư mục optuna_workspace
MULTI_AGENTS_PATH = os.path.join(ROOT_DIR, 'multiAgents.py')
TEMP_AGENTS_PATH = os.path.join(ROOT_DIR, 'temp_q6_Agents.py')
DB_PATH = "sqlite:///q6_tuning_v5.db"

NUM_GAMES = 30
MAP_NAME = "smallClassic"
NUM_GHOSTS = 1


In [5]:
def objective(trial):
    # Khai báo các tham số cần tune cho code mới (ít tham số hơn)
    w_food_dist = trial.suggest_float("w_food_dist", 0.1, 2000.0)
    w_food_rem = trial.suggest_float("w_food_rem", 0.1, 2000.0)
    w_capsule_rem = trial.suggest_float("w_capsule_rem", 0.1, 2000.0)
    w_scared_ghost = trial.suggest_float("w_scared_ghost", 0.1, 2000.0)
    w_death = trial.suggest_float("w_death", 0.1, 2000.0)
    w_active_ghost = trial.suggest_float("w_active_ghost", 0.1, 2000.0)
    w_entrapment = trial.suggest_float("w_entrapment", 0.1, 2000.0)
    w_escape = trial.suggest_float("w_escape", 0.1, 2000.0)
    
    dof_radius = trial.suggest_int("dof_radius", 1, 10)
    threat_radius = trial.suggest_int("threat_radius", 1, 10)

    # Đọc code từ multiAgents.py
    with open(MULTI_AGENTS_PATH, 'r', encoding='utf-8') as f:
        code = f.read()

    # Đổi tên class để tránh xung đột
    code = code.replace("class ExpectimaxAgent(MultiAgentSearchAgent):", "class OptunaAgent(MultiAgentSearchAgent):")
    
    # Tiêm trọng số bằng regex chính xác cho multiAgents.py mới
    code = re.sub(r"food_distance_bonus = [\d\.]+", f"food_distance_bonus = {w_food_dist}", code)
    code = re.sub(r"food_remaining_penalty = [\d\.]+ \*", f"food_remaining_penalty = {w_food_rem} *", code)
    code = re.sub(r"capsule_penalty = [\d\.]+ \*", f"capsule_penalty = {w_capsule_rem} *", code)
    code = re.sub(r"scared_ghost_bonus = [\d\.]+", f"scared_ghost_bonus = {w_scared_ghost}", code)
    code = re.sub(r"score -= [\d\.]+\s+# Imminent death", f"score -= {w_death}  # Imminent death", code)
    code = re.sub(r"active_ghost_penalty = [\d\.]+ /", f"active_ghost_penalty = {w_active_ghost} /", code)
    code = re.sub(r"entrapment_risk = ghost_threat \* narrowness_factor \* [\d\.]+", f"entrapment_risk = ghost_threat * narrowness_factor * {w_entrapment}", code)
    code = re.sub(r"escape_bonus = [\d\.]+ \*", f"escape_bonus = {w_escape} *", code)
    
    code = re.sub(r"DOF_RADIUS = \d+", f"DOF_RADIUS = {dof_radius}", code)
    code = re.sub(r"THREAT_RADIUS = \d+", f"THREAT_RADIUS = {threat_radius}", code)

    # Lưu ra file tạm
    with open(TEMP_AGENTS_PATH, 'w', encoding='utf-8') as f:
        f.write(code)

    scores = []
    
    try:
        # Chạy game pacman
        cmd = [
            sys.executable, "pacman.py", 
            "-p", "OptunaAgent", 
            "-a", "evalFn=riskAware", 
            "-l", MAP_NAME, 
            "-k", str(NUM_GHOSTS), 
            "-q", 
            "-n", str(NUM_GAMES)
        ]
        out = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT_DIR, timeout=300)
        
        output_str = out.stdout
        
        # Tìm các điểm số: Scores:       100.0, 200.0, 300.0
        score_match = re.search(r'Scores:\s+([\d\.\-\, ]+)', output_str)
        if score_match:
            scores_str = score_match.group(1)
            scores = [float(s.strip()) for s in scores_str.split(',')]
            
        if not scores:
            return -5000.0 # Bị lỗi không lấy được điểm
            
        # Tính điểm trung vị
        median_score = statistics.median(scores)
        
        # In ra log cho từng trial
        print(f"Trial {trial.number}: Median Score: {median_score:.2f} | Scores: {scores}")
        
        return median_score
        
    except subprocess.TimeoutExpired:
        print(f"Trial {trial.number}: Timeout!")
        return -5000.0
    except Exception as e:
        print(f"Trial {trial.number}: Error - {e}")
        return -5000.0
    finally:
        # Dọn dẹp
        if os.path.exists(TEMP_AGENTS_PATH):
            os.remove(TEMP_AGENTS_PATH)
        pyc = TEMP_AGENTS_PATH + 'c'
        if os.path.exists(pyc): 
            os.remove(pyc)


In [6]:
# Tạo hoặc load study
study_name = "q6_tuning_smallclassic_1ghost_v5"
study = optuna.create_study(
    study_name=study_name,
    storage=DB_PATH,
    direction="maximize",
    load_if_exists=True
)

# Chạy tối ưu hóa
study.optimize(objective, n_trials=500)

print(f"Giá trị Median tốt nhất: {study.best_value}")
print("Bộ tham số tốt nhất:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")


[I 2026-05-18 11:00:53,359] A new study created in RDB with name: q6_tuning_smallclassic_1ghost_v5
[I 2026-05-18 11:02:45,001] Trial 0 finished with value: -146.0 and parameters: {'w_food_dist': 91.60527490580057, 'w_food_rem': 1462.1305110009366, 'w_capsule_rem': 1347.7054823197063, 'w_scared_ghost': 1695.8374093821972, 'w_death': 525.6076268206109, 'w_active_ghost': 1610.7158583402745, 'w_entrapment': 1201.1534663857854, 'w_escape': 1023.7777920982248, 'dof_radius': 2, 'threat_radius': 8}. Best is trial 0 with value: -146.0.


Trial 0: Median Score: -146.00 | Scores: [-155.0, -137.0, -671.0, -3529.0, -222.0, -791.0, -581.0, -223.0, -19.0, 916.0, 840.0, -166.0, -1693.0, -215.0, 913.0, 929.0, 915.0, 961.0, 881.0, 734.0, -221.0, -5169.0, 943.0, -158.0, -135.0, 743.0, 839.0, -891.0, 908.0, -200.0]


[I 2026-05-18 11:03:46,321] Trial 1 finished with value: 701.5 and parameters: {'w_food_dist': 484.03024563912874, 'w_food_rem': 286.412984117793, 'w_capsule_rem': 1589.5232413135163, 'w_scared_ghost': 1372.822194881443, 'w_death': 1606.6634204779996, 'w_active_ghost': 1067.5774598345113, 'w_entrapment': 53.83763349640687, 'w_escape': 525.1223539478921, 'dof_radius': 2, 'threat_radius': 4}. Best is trial 1 with value: 701.5.


Trial 1: Median Score: 701.50 | Scores: [701.0, 769.0, 691.0, 848.0, 526.0, 825.0, 476.0, 805.0, 819.0, 1.0, 497.0, 314.0, -240.0, 763.0, 350.0, 659.0, 775.0, 702.0, 1108.0, 851.0, 651.0, 720.0, 1017.0, 918.0, 367.0, 831.0, -585.0, 906.0, 635.0, 609.0]


[I 2026-05-18 11:03:50,069] Trial 2 finished with value: -447.0 and parameters: {'w_food_dist': 1023.0258158510624, 'w_food_rem': 1975.4323573584732, 'w_capsule_rem': 1782.3863739716812, 'w_scared_ghost': 301.97205440375535, 'w_death': 1864.4364763161025, 'w_active_ghost': 492.9953075428954, 'w_entrapment': 852.4667029919823, 'w_escape': 1801.3341669073106, 'dof_radius': 6, 'threat_radius': 10}. Best is trial 1 with value: 701.5.


...

Trial 497: Median Score: 1174.00 | Scores: [1177.0, 1177.0, 975.0, 1373.0, 1373.0, 1162.0, -70.0, 1373.0, 1163.0, 1165.0, 1370.0, 1163.0, 1167.0, 1175.0, 1170.0, 1345.0, 1360.0, 1172.0, 1371.0, 976.0, 1370.0, 1173.0, 1357.0, 975.0, 1363.0, 1371.0, 127.0, 1171.0, 1374.0, 1170.0]


[I 2026-05-18 14:13:33,156] Trial 498 finished with value: 1170.0 and parameters: {'w_food_dist': 1412.7794255105634, 'w_food_rem': 729.5559958364557, 'w_capsule_rem': 1579.7770789516412, 'w_scared_ghost': 150.1785017120607, 'w_death': 872.5669949696204, 'w_active_ghost': 418.1317045255102, 'w_entrapment': 1317.9896043606243, 'w_escape': 1.4438858844607125, 'dof_radius': 1, 'threat_radius': 1}. Best is trial 269 with value: 1359.5.


Trial 498: Median Score: 1170.00 | Scores: [1161.0, 975.0, 977.0, 1373.0, 1372.0, 1173.0, 971.0, 1171.0, 1177.0, 1171.0, 1169.0, 1359.0, 1362.0, 1173.0, 1182.0, 1175.0, 982.0, -78.0, 985.0, 975.0, 1167.0, 1167.0, 1168.0, 1366.0, 977.0, 1167.0, 972.0, 1357.0, 1171.0, 1365.0]


[I 2026-05-18 14:13:45,317] Trial 499 finished with value: 1167.5 and parameters: {'w_food_dist': 1356.6571829860495, 'w_food_rem': 699.8173485697085, 'w_capsule_rem': 1613.072376224358, 'w_scared_ghost': 182.18965155923846, 'w_death': 930.5145066632208, 'w_active_ghost': 377.66631004153, 'w_entrapment': 1246.9361592529644, 'w_escape': 42.55398881408013, 'dof_radius': 1, 'threat_radius': 1}. Best is trial 269 with value: 1359.5.


Trial 499: Median Score: 1167.50 | Scores: [1170.0, 1173.0, 1364.0, 973.0, 1171.0, 1369.0, 1372.0, 969.0, 1175.0, 1168.0, 1167.0, 1165.0, 955.0, 978.0, 1173.0, 1166.0, 1164.0, 1161.0, 1168.0, 1150.0, 1171.0, 1371.0, 1172.0, 1374.0, 970.0, 1166.0, 1174.0, 977.0, 975.0, 971.0]
Giá trị Median tốt nhất: 1359.5
Bộ tham số tốt nhất:
    w_food_dist: 1425.6101897184901
    w_food_rem: 607.9938417289302
    w_capsule_rem: 1377.4453441979385
    w_scared_ghost: 246.54090600924965
    w_death: 995.6589916451169
    w_active_ghost: 476.8072229368562
    w_entrapment: 1249.18564742111
    w_escape: 49.22807989033885
    dof_radius: 1
    threat_radius: 1


In [7]:
# Trực quan hóa
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice
try:
    study
except NameError:
    study = optuna.load_study(study_name="q6_tuning_smallclassic_1ghost_v5", storage="sqlite:///q6_tuning_v5.db")

# 1. Biểu đồ lịch sử tối ưu
plot_optimization_history(study).show()


In [8]:
# 2. Biểu đồ độ quan trọng của tham số
plot_param_importances(study).show()


In [9]:
# 3. Biểu đồ lát cắt
plot_slice(study).show()


In [ ]:
# In ra điểm số cao nhất (Best Value)
print("🏆 Điểm Median cao nhất đạt được:", study.best_value)

# Duyệt và in ra toàn bộ các thông số (Best Params)
print("\n🔥 Bộ thông số hoàn hảo nhất:")
for key, value in study.best_params.items():
    print(f"    {key} = {value}")

🏆 Điểm Median cao nhất đạt được: 1359.5

🔥 Bộ thông số hoàn hảo nhất:
    w_food_dist = 1425.6101897184901
    w_food_rem = 607.9938417289302
    w_capsule_rem = 1377.4453441979385
    w_scared_ghost = 246.54090600924965
    w_death = 995.6589916451169
    w_active_ghost = 476.8072229368562
    w_entrapment = 1249.18564742111
    w_escape = 49.22807989033885
    dof_radius = 1
    threat_radius = 1
